
### ランダム変数の影響

ある説明変数特徴量がランダム変数より影響が大きいことが「重要」であることを示す
一つの指標になりえます。


In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)
# warning messageを出さない
warnings.filterwarnings('ignore')

import os
os.makedirs("image_executed", exist_ok=True)

In [ ]:
def get_data():
    """load data from a file.

    Returns:
        pd.DataFrame: data.
        [str] : a list of explanatory variable names.
        str: target name.
    """
    df = pd.read_csv("../data/TC_ReCo_detail_descriptor.csv")

    descriptor_names = ['C_R', 'C_T', 'vol_per_atom', 'Z', 'f4', 'd5', 'L4f', 'S4f', 'J4f',
                        '(g-1)J4f', '(2-g)J4f']
    target_name = 'Tc'
    print("descriptor=", descriptor_names)
    print("target=", target_name)
    return df, descriptor_names, target_name


g_df, g_descriptor_names, g_target_name = get_data()


In [ ]:
from sklearn.preprocessing import StandardScaler


def get_Xy(df, descriptor_names, target_name):
    """get X and y from dataframe.
    
    column 'random', ~N(0,1), is added to dataframe.

    Args:
        df (pd.DataFrame): data.
        descriptor_names ([str]): a list of explanatory variable names.
        target_name (str): target variable name

    Returns:
        np.ndarray: X
        np.ndarray: y
        [str]: a list of explanatory variable names, where 'random' is added.
    """
    np.random.seed(1)
    df_std = df[descriptor_names].copy()
    scaler = StandardScaler()
    df_std.iloc[:, :] = scaler.fit_transform(df_std.values)
    df_std["random"] = np.random.normal(0, 1, size=df_std.shape[0])
    descriptor_names = list(df_std.columns)
    X = df_std.values
    y = df[target_name].values
    return X, y, descriptor_names


g_X, g_y, g_descriptor_names = get_Xy(g_df, g_descriptor_names, g_target_name)
# random is added to descriptor_names


In [ ]:
g_descriptor_names

In [ ]:
def show_X(X):
    fig, ax = plt.subplots()
    ax.plot(X)
    ax.set_xlabel("index")
    ax.set_ylabel("x")
    fig.show()


show_X(g_X)


def show_hist(y):
    fig, ax = plt.subplots()
    ax.hist(y)
    ax.set_xlabel("y")
    fig.show()


show_hist(g_y)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split


In [ ]:
def get_model(regname: str, X: np.ndarray, y: np.ndarray):
    """get a regression model.

    Args:
        regname (str): regression model name.
        X (np.ndarray): X
        y (np.ndarray): y

    Returns:
        RandomForestRegressor|RidgeCV: regression model.
    """
    if regname == "RF":
        reg = RandomForestRegressor(n_estimators=100)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.8)
        reg.fit(X_train, y_train)

    elif regname == "RidgeCV":
        kf = KFold(n_splits=10, shuffle=True)
        reg = RidgeCV(cv=kf)
        reg.fit(X, y)
    return reg


g_regname = "RF"  # RF or RidgeCV
g_reg = get_model(g_regname, g_X, g_y)


In [ ]:
def show_rf_importance(reg, descriptor_names):
    """show feature importance.

    Args:
        reg ([type]): [description]
        descriptor_names ([type]): [description]
    """
    try:  # tryはRidgeではreg.feature_importances_が存在しないからエラーを起こしても止まらないようにするため。
        reg.feature_importances_
        df_rf_imp = pd.DataFrame(
            {"label": descriptor_names, "importance": reg.feature_importances_})
        df_rf_imp.sort_values(by="importance", ascending=False, inplace=True)
        df_rf_imp.plot.bar(x="label", y="importance",)
    except:
        pass


show_rf_importance(g_reg, g_descriptor_names)


In [ ]:
from sklearn.inspection import permutation_importance
g_feature_importance = permutation_importance(
    g_reg, g_X, g_y, n_repeats=30, random_state=20)
g_df_perm = pd.DataFrame(
    g_feature_importance["importances"], index=g_descriptor_names).T


In [ ]:
def show_r2_decrease(df):
    """show LOO-R2 values.

    Args:
        df (pd.DataFrame): data.
    """
    score_mean = np.mean(df.values, axis=0)
    iorder = np.argsort(score_mean)[::-1]
    fig, ax = plt.subplots()
    df.iloc[:, iorder].boxplot(rot=90, ax=ax)
    ax.set_ylabel("$R^2$ decrease")
    fig.tight_layout()
    fig.savefig("image_executed/RECo_RF_random_permutation_importance.png")
    fig.show()


show_r2_decrease(g_df_perm)


#### 問題1

- random stateを変える。

- RFをRidgeCVに変更する。
#### 問題２

データを変える。
